In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
import utils
import seaborn as sns
import pingouin as pg
import matplotlib.cm as cm

from statsmodels.stats.anova import AnovaRM
from pathlib import Path

from natsort import index_natsorted

path_figs = "./Figs/"

fingers = ['1', '2', '3', '4', '5']

iti = utils.iti
planTime = utils.planTime
feedbackTime = utils.feedbackTime

total_sub_num = utils.total_sub_num
num_days = utils.num_days
num_blocks_per_day = utils.num_blocks_per_day
num_trials_per_block = utils.num_trials_per_block
conditions = utils.conditions

# Fill in after collection, or leave None to use every .dat file found
sub_nums = None

utils.set_figure_style("1col")
sns.color_palette('colorblind')


[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [2]:
# reload utils
import importlib
importlib.reload(utils)


<module 'utils' from '/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/SingleFingerGainGeneralization/utils.py'>

`.dat` files live in `Single_Finger_Gain_Generalization/` (copied from MotorControl `data/EFC_learningDynamics/CAD_Spat_Vis/Single_Finger_Gain_Generalization/`).

Expected names: `single_gain_generalization_<subNum>.dat` (same convention as Version 3).


In [3]:
available = utils.list_dat_subjects()
print("Found .dat subjects:", available)

if sub_nums is None:
    sub_nums = available

if not sub_nums:
    raise FileNotFoundError(
        f"No .dat files in {utils.dat_dir().resolve()}. "
        "Data is not acquired yet. After collection, copy "
        "single_gain_generalization_<subNum>.dat into that folder and re-run."
    )

missing = [s for s in sub_nums if s not in available]
if missing:
    raise FileNotFoundError(f"Missing .dat files for subjects: {missing}")

subjs_list = utils.read_dat_files_subjs_list(sub_nums)
subjs = pd.concat(subjs_list, ignore_index=True)
subjs = subjs.sort_values(['subNum', 'BN', 'TN']).reset_index(drop=True)

subjs = utils.add_total_trial_num(subjs)
subjs = utils.add_day(subjs)
subjs['chord'] = subjs.apply(utils.extract_chord_form_target_force, axis=1)
subjs['condition'] = subjs.apply(utils.extract_condition, axis=1)
subjs['num_targets'] = subjs[[f'targetForce{i}' for i in range(1, 6)]].ne(0).sum(axis=1)
subjs['count'] = 1
subjs = utils.add_transitions(subjs)
subjs.reset_index(drop=True, inplace=True)


Found .dat subjects: [1, 2, 3]


In [4]:
subjs.columns


Index(['BN', 'TN', 'subNum', 'targetForce1', 'targetForce2', 'targetForce3',
       'targetForce4', 'targetForce5', 'endForce1', 'endForce2', 'endForce3',
       'endForce4', 'endForce5', 'endForcePurturbed1', 'endForcePurturbed2',
       'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
       'isTargetVisible', 'purturbation1', 'purturbation2', 'planTime',
       'feedbackTime', 'iti', 'forceGain', 'trialCorr', 'trialErrorType',
       'TotalTrialNum', 'day', 'chord', 'condition', 'num_targets', 'count',
       'prev_condition', 'prev_perturbation', 'prev_trialCorr', 'transition',
       'trial_num_within_condition'],
      dtype='str')

In [5]:
subjs = subjs[['subNum', 'BN', 'TN', 'TotalTrialNum', 'trial_num_within_condition',
            'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5',
            'endForce1', 'endForce2', 'endForce3', 'endForce4', 'endForce5', 'isTargetVisible',
            'endForcePurturbed1', 'endForcePurturbed2', 'endForcePurturbed3', 'endForcePurturbed4', 'endForcePurturbed5',
            'purturbation1', 'purturbation2', 'forceGain', 'trialCorr', 'trialErrorType', 'num_targets',
            'chord', 'condition', 'prev_condition', 'prev_perturbation', 'prev_trialCorr', 'transition', 'day', 'count']]


In [6]:
subjs.head()


,subNum,BN,TN,TotalTrialNum,trial_num_within_condition,targetForce1,targetForce2,targetForce3,targetForce4,targetForce5,...,trialErrorType,num_targets,chord,condition,prev_condition,prev_perturbation,prev_trialCorr,transition,day,count
0,1,1,1,1,1,0.0,1.5,0.0,0.0,0.0,...,0,1,-^---,1.5,NaN,NaN,NaN,NaN,1,1
1,1,1,2,2,1,0.0,-3.0,0.0,0.0,0.0,...,0,1,-v---,-3.0,1.5,0.81,1.0,1.5->-3.0,1,1
2,1,1,3,3,2,0.0,-3.0,0.0,0.0,0.0,...,0,1,-v---,-3.0,-3.0,-0.31,1.0,-3.0->-3.0,1,1
3,1,1,4,4,1,0.0,3.0,0.0,0.0,0.0,...,0,1,-^---,3.0,-3.0,-0.26,1.0,-3.0->3.0,1,1
4,1,1,5,5,1,0.0,-1.5,0.0,0.0,0.0,...,0,1,-v---,-1.5,3.0,-0.54,1.0,3.0->-1.5,1,1


In [7]:
print(subjs.groupby('subNum').agg(n_trials=('TN', 'count'), n_blocks=('BN', 'nunique'), n_correct=('trialCorr', 'sum')))
print(subjs['condition'].value_counts().sort_index())
print(subjs['day'].value_counts().sort_index())


        n_trials  n_blocks  n_correct
subNum                               
1            320        10        312
2            641        20        599
3            320        10        312
condition
-3.0    310
-1.5    313
 1.5    329
 3.0    329
Name: count, dtype: int64
day
1    960
2    321
Name: count, dtype: int64


In [8]:
Path(utils.path_misc).mkdir(parents=True, exist_ok=True)
subjs.to_csv(utils.path_misc + 'subjs.csv', index=False, sep='\t')
print('saved', utils.path_misc + 'subjs.csv')


saved ./Single_Finger_Gain_Generalization_miscs/subjs.csv
